In [1]:
!apt-get update -qq
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
!tar xf spark-3.5.1-bin-hadoop3.tgz
!pip install -q pyspark==3.5.1 graphframes


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 1.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 9.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.0.2 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.1 which is incompatible.


In [2]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"

In [3]:
from pyspark.sql import SparkSession

# Use version 0.8.4 which is optimized for Spark 3.5 and Scala 2.12
spark = SparkSession.builder \
    .appName("GraphFramesExample") \
    .config("spark.jars.packages", "graphframes:graphframes:0.8.4-spark3.5-s_2.12") \
    .getOrCreate()

# Verify versions
print("Spark Version:", spark.version)

Spark Version: 3.5.1


In [4]:
from graphframes import GraphFrame

v_data = [
    ("1", "Alice", "IT"),
    ("2", "Bob", "IT"),
    ("3", "Charlie", "HR"),
    ("4", "David", "Marketing"),
    ("5", "Eve", "IT"),
    ("6", "Frank", "HR")
]
vertices = spark.createDataFrame(v_data, ["id", "name", "department"])

# ৪. list of relation (Edges) [cite: 4, 11, 139]
e_data = [
    ("1", "2", "mentor"),      # A mentors B
    ("2", "3", "mentor"),      # B mentors C
    ("3", "1", "mentor"),      # C mentors A (Circle)
    ("4", "2", "reports_to"),  # D reports to B
    ("2", "5", "reports_to"),  # B reports to E (Chain)
    ("1", "5", "colleague"),   # Alice and Eve are colleagues
    ("5", "1", "colleague"),   # Reverse colleague
    ("5", "6", "mentor"),      # Eve mentors Frank
    ("4", "1", "mentor")       # David mentors Alice
]
edges = spark.createDataFrame(e_data, ["src", "dst", "relationship"])

# ৫. graph builder [cite: 142]
g = GraphFrame(vertices, edges)
print("Vertices:")
g.vertices.show()

print("Edges:")
g.edges.show()

print("Degrees:")
g.degrees.show()

print("In-Degrees:")
g.inDegrees.show()

print("Out-Degrees:")
g.outDegrees.show()

Vertices:
+---+-------+----------+
| id|   name|department|
+---+-------+----------+
|  1|  Alice|        IT|
|  2|    Bob|        IT|
|  3|Charlie|        HR|
|  4|  David| Marketing|
|  5|    Eve|        IT|
|  6|  Frank|        HR|
+---+-------+----------+

Edges:
+---+---+------------+
|src|dst|relationship|
+---+---+------------+
|  1|  2|      mentor|
|  2|  3|      mentor|
|  3|  1|      mentor|
|  4|  2|  reports_to|
|  2|  5|  reports_to|
|  1|  5|   colleague|
|  5|  1|   colleague|
|  5|  6|      mentor|
|  4|  1|      mentor|
+---+---+------------+

Degrees:
+---+------+
| id|degree|
+---+------+
|  3|     2|
|  1|     5|
|  4|     2|
|  2|     4|
|  5|     4|
|  6|     1|
+---+------+

In-Degrees:
+---+--------+
| id|inDegree|
+---+--------+
|  3|       1|
|  1|       3|
|  2|       2|
|  5|       2|
|  6|       1|
+---+--------+

Out-Degrees:
+---+---------+
| id|outDegree|
+---+---------+
|  3|        1|
|  1|        2|
|  4|        2|
|  2|        2|
|  5|        2|
+--

In [5]:
# Task1
tri_pattern = g.find("(a)-[e1]->(b); (b)-[e2]->(c); (c)-[e3]->(a)") \
    .filter("e1.relationship = 'mentor' AND e2.relationship = 'mentor' AND e3.relationship = 'mentor'")
tri_pattern.select("a.name", "b.name", "c.name").show()

+-------+-------+-------+
|   name|   name|   name|
+-------+-------+-------+
|Charlie|  Alice|    Bob|
|    Bob|Charlie|  Alice|
|  Alice|    Bob|Charlie|
+-------+-------+-------+



In [6]:
#Task2
chain = g.find("(a)-[e1]->(b); (b)-[e2]->(c)") \
    .filter("e1.relationship = 'reports_to' AND e2.relationship = 'reports_to'")
chain.select("a.name", "b.name", "c.name").show()

+-----+----+----+
| name|name|name|
+-----+----+----+
|David| Bob| Eve|
+-----+----+----+



In [7]:
# Task3
mentors_ab = g.find("(a)-[e1]->(b)").filter("e1.relationship = 'mentor'")
mentors_ba = g.find("(b)-[e2]->(a)").filter("e2.relationship = 'mentor'")

one_way = mentors_ab.join(mentors_ba, ["a", "b"], "left_anti")
one_way.select("a.name", "b.name").show()

+-------+-------+
|   name|   name|
+-------+-------+
|Charlie|  Alice|
|  Alice|    Bob|
|    Bob|Charlie|
|  David|  Alice|
|    Eve|  Frank|
+-------+-------+



In [8]:
# Task4
mentor_counts = g.edges.filter("relationship = 'mentor'") \
    .groupBy("dst").count() \
    .withColumnRenamed("dst", "id").withColumnRenamed("count", "total_mentees")

mentor_counts.join(g.vertices, "id").orderBy("total_mentees", ascending=False).show(1)

+---+-------------+-----+----------+
| id|total_mentees| name|department|
+---+-------------+-----+----------+
|  1|            2|Alice|        IT|
+---+-------------+-----+----------+
only showing top 1 row



In [9]:
# Task5
colleague_g = GraphFrame(vertices, edges.filter("relationship = 'colleague'"))
colleague_g.degrees.join(vertices, "id").orderBy("degree", ascending=True).show()

+---+------+-----+----------+
| id|degree| name|department|
+---+------+-----+----------+
|  1|     2|Alice|        IT|
|  5|     2|  Eve|        IT|
+---+------+-----+----------+



In [10]:
# Task6
communities = g.labelPropagation(maxIter=5)
communities.select("id", "name", "department", "label").show()

+---+-------+----------+-------------+
| id|   name|department|        label|
+---+-------+----------+-------------+
|  3|Charlie|        HR|1623497637888|
|  5|    Eve|        IT|1623497637888|
|  6|  Frank|        HR| 420906795008|
|  1|  Alice|        IT| 420906795008|
|  4|  David| Marketing|1623497637888|
|  2|    Bob|        IT| 420906795008|
+---+-------+----------+-------------+



In [11]:
# Task7
pr = g.pageRank(resetProbability=0.15, tol=0.01)
pr.vertices.select("id", "name", "pagerank").orderBy("pagerank", ascending=False).show(5)

+---+-------+------------------+
| id|   name|          pagerank|
+---+-------+------------------+
|  1|  Alice|1.5982829217886474|
|  5|    Eve|1.4143360511735508|
|  2|    Bob|1.0804378060307018|
|  6|  Frank|0.8795019129708735|
|  3|Charlie|0.7414441686987516|
+---+-------+------------------+
only showing top 5 rows

